# Notebook 03 — Phase-Lock Validation

**Repo:** `temporal-phase-lock`  
**Notebook:** `notebooks/03_phase_lock_validation.ipynb`  
**Purpose:** measure whether adaptive temporal reasoning remains stable across decomposition, evidence, review, and answer stages.

Notebook 01 established a baseline reasoning scaffold.  
Notebook 02 decomposed temporal structure into anchors, operators, timelines, and graph edges.  
Notebook 03 adds the core contribution:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```

This becomes the 45° phase-lock gate.

## Core pipeline

```text
case
  → decomposition features
  → evidence features
  → review features
  → answer features
  → cosine alignment
  → 45° phase-lock gate
  → stable / near-boundary / drift label
```

## 0. Environment setup

In [ ]:
import os, re, math, json, random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

REPO_NAME = "temporal-phase-lock"
NOTEBOOK_ID = "03_phase_lock_validation"

BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
DOCS_DIR = BASE_DIR / "docs"
ARTIFACT_DIR = BASE_DIR / "artifacts"

for d in [FIG_DIR, DOCS_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE_LOCK_GATE = 1 / math.sqrt(1**2 + 1**2)
NEAR_BOUNDARY_MARGIN = 0.05

print(f"Ready: {REPO_NAME}/{NOTEBOOK_ID}")
print(f"Phase-lock gate: {PHASE_LOCK_GATE:.6f}")

## 1. Recreate Notebook 01/02 cases

In [ ]:
@dataclass
class TemporalCase:
    case_id: str
    question: str
    evidence: List[str]
    gold: str
    answer_type: str

CASES = [
    TemporalCase("TQA-001","Who joined after Mira but before Theo?",
        ["Mira joined the project in March.","Jon joined the project in May.","Theo joined the project in August.","Leah joined the project in November."],
        "Jon","between"),
    TemporalCase("TQA-002","Which event happened first?",
        ["The calibration finished on Tuesday.","The sensor test started on Monday.","The report was drafted on Wednesday."],
        "sensor test","first"),
    TemporalCase("TQA-003","What happened immediately after the review?",
        ["The draft happened before the review.","The review happened before the revision.","The revision happened before the release."],
        "revision","after"),
    TemporalCase("TQA-004","Which item was completed last?",
        ["Notebook 01 was completed before Notebook 02.","Notebook 03 was completed after Notebook 02.","Notebook 04 was completed after Notebook 03."],
        "Notebook 04","last"),
    TemporalCase("TQA-005","Did the audit occur before deployment?",
        ["The prototype was tested in April.","The audit occurred in June.","Deployment occurred in September."],
        "Yes","yes_no"),
]

case_df = pd.DataFrame([asdict(c) for c in CASES])
case_df

## 2. Shared temporal parsing utilities

In [ ]:
MONTH_ORDER = {"january":1,"february":2,"march":3,"april":4,"may":5,"june":6,"july":7,"august":8,"september":9,"october":10,"november":11,"december":12}
WEEKDAY_ORDER = {"monday":1,"tuesday":2,"wednesday":3,"thursday":4,"friday":5,"saturday":6,"sunday":7}

def clean_phrase(text: str) -> str:
    text = re.sub(r"[\?\.]$", "", text.strip())
    text = re.sub(r"^(the|a|an)\s+", "", text, flags=re.I)
    return text

def extract_subject(sentence: str) -> str:
    parts = re.split(r"\s+(joined|occurred|finished|started|was|were|completed|happened|drafted|tested)\b",
                     sentence, maxsplit=1, flags=re.I)
    return clean_phrase(parts[0])

def detect_operators(question: str) -> List[str]:
    q = question.lower()
    ops = []
    if q.startswith("did") and "before" in q: ops.append("yes_no_before")
    if "after" in q and "before" in q: ops.append("between")
    if "immediately after" in q: ops.append("immediately_after")
    if "happened first" in q or "event happened first" in q or " first" in q: ops.append("first")
    if "completed last" in q or " last" in q: ops.append("last")
    for op in ["immediately after", "after", "before", "between", "first", "last"]:
        tag = op.replace(" ", "_")
        if op in q and tag not in ops:
            ops.append(tag)
    return list(dict.fromkeys(ops))

def extract_question_anchors(question: str) -> Dict:
    q = question.strip()
    ql = q.lower()
    m = re.search(r"after\s+([A-Za-z0-9 ]+?)\s+but\s+before\s+([A-Za-z0-9 ]+)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"entity_between_anchors","relation":"after_left_and_before_right","required_anchor_count":2}
    m = re.search(r"immediately after\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1))],
                "target":"successor_event","relation":"immediate_successor","required_anchor_count":1}
    m = re.match(r"did\s+(.+?)\s+occur\s+before\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"yes_no_comparison","relation":"before","required_anchor_count":2}
    if "first" in ql:
        return {"anchors":["timeline_start"],"target":"earliest_event","relation":"min_time","required_anchor_count":1}
    if "last" in ql:
        return {"anchors":["timeline_end"],"target":"latest_event","relation":"max_time","required_anchor_count":1}
    return {"anchors":[],"target":"unknown","relation":"unknown","required_anchor_count":0}

def extract_time_fact(sentence: str):
    s = sentence.lower()
    for month, idx in MONTH_ORDER.items():
        if month in s:
            return extract_subject(sentence), idx, month.title()
    for day, idx in WEEKDAY_ORDER.items():
        if day in s:
            return extract_subject(sentence), idx, day.title()
    return None

def extract_directed_edges(sentence: str):
    s = sentence.strip().rstrip(".")
    patterns = [
        (r"(.+?)\s+was completed before\s+(.+)", False),
        (r"(.+?)\s+was completed after\s+(.+)", True),
        (r"(.+?)\s+happened before\s+(.+)", False),
        (r"(.+?)\s+happened after\s+(.+)", True),
    ]
    for pat, reverse in patterns:
        m = re.match(pat, s, flags=re.I)
        if m:
            a, b = clean_phrase(m.group(1)), clean_phrase(m.group(2))
            return [(b, "before", a)] if reverse else [(a, "before", b)]
    return []

def topological_order_from_edges(edges):
    nodes = sorted(set([a for a,_,b in edges] + [b for a,_,b in edges]))
    incoming = {n:set() for n in nodes}
    outgoing = {n:set() for n in nodes}
    for a, rel, b in edges:
        if rel == "before":
            outgoing[a].add(b); incoming[b].add(a)
    order, ready = [], sorted([n for n in nodes if not incoming[n]])
    while ready:
        n = ready.pop(0); order.append(n)
        for m in sorted(outgoing[n]):
            incoming[m].discard(n)
            if not incoming[m]: ready.append(m)
        ready = sorted(set(ready))
    return order

def decompose_evidence(case: TemporalCase) -> Dict:
    facts, labels, explicit_edges, rows = {}, {}, [], []
    for sent in case.evidence:
        tf = extract_time_fact(sent)
        if tf:
            subject, idx, label = tf
            facts[subject] = idx
            labels[subject] = label
            rows.append({"source": subject, "relation": "has_time", "target": label, "edge_type": "fact"})
        for a, rel, b in extract_directed_edges(sent):
            explicit_edges.append((a, rel, b))
            rows.append({"source": a, "relation": rel, "target": b, "edge_type": "constraint"})
    derived_edges = []
    if facts:
        timeline = [k for k,v in sorted(facts.items(), key=lambda kv: kv[1])]
        for a, b in zip(timeline[:-1], timeline[1:]):
            derived_edges.append((a, "before", b))
            rows.append({"source": a, "relation": "before", "target": b, "edge_type": "derived"})
    else:
        timeline = topological_order_from_edges(explicit_edges) if explicit_edges else []
    return {"facts": facts, "labels": labels, "explicit_edges": explicit_edges,
            "derived_edges": derived_edges, "all_edges": explicit_edges + derived_edges,
            "timeline": timeline, "rows": rows}

def loose_contains(name: str, collection: List[str]) -> bool:
    n = name.lower().strip()
    return any(n in item.lower() or item.lower() in n for item in collection)

## 3. Recreate baseline answers and review features

In [ ]:
def reformulate(question: str) -> Dict:
    ops = detect_operators(question)
    if "between" in ops: return {"intent":"between"}
    if "first" in ops: return {"intent":"first"}
    if "immediately_after" in ops: return {"intent":"after"}
    if "last" in ops: return {"intent":"last"}
    if "yes_no_before" in ops: return {"intent":"yes_no_before"}
    return {"intent":"unknown"}

def answer_from_trace(case: TemporalCase, rewritten: Dict) -> str:
    intent = reformulate(case.question)["intent"]
    timeline = rewritten["timeline"]
    facts = rewritten["facts"]
    if intent == "between":
        anchors = extract_question_anchors(case.question)["anchors"]
        if len(anchors) == 2 and anchors[0] in facts and anchors[1] in facts:
            between = [name for name, t in facts.items() if facts[anchors[0]] < t < facts[anchors[1]]]
            return between[0] if between else "UNKNOWN"
        return "UNKNOWN"
    if intent == "first":
        return timeline[0] if timeline else "UNKNOWN"
    if intent == "after":
        matches = [i for i, item in enumerate(timeline) if "review" in item.lower()]
        if matches and matches[0] + 1 < len(timeline):
            return timeline[matches[0] + 1]
        return "UNKNOWN"
    if intent == "last":
        return timeline[-1] if timeline else "UNKNOWN"
    if intent == "yes_no_before":
        anchors = extract_question_anchors(case.question)["anchors"]
        if len(anchors) == 2:
            a_key = next((k for k in facts if anchors[0].lower() in k.lower() or k.lower() in anchors[0].lower()), None)
            b_key = next((k for k in facts if anchors[1].lower() in k.lower() or k.lower() in anchors[1].lower()), None)
            if a_key and b_key:
                return "Yes" if facts[a_key] < facts[b_key] else "No"
        return "UNKNOWN"
    return "UNKNOWN"

def review_answer(pred: str, gold: str, rewritten: Dict) -> Dict:
    pred_norm = pred.lower().strip()
    gold_norm = gold.lower().strip()
    exact = pred_norm == gold_norm or pred_norm in gold_norm or gold_norm in pred_norm
    supported = pred != "UNKNOWN" and (pred in rewritten["timeline"] or pred in rewritten["facts"] or pred in ["Yes", "No"])
    return {
        "exact_match": int(bool(exact)),
        "supported": int(bool(supported)),
        "answer_known": int(pred != "UNKNOWN"),
        "review_status": "PASS" if exact and supported else "CHECK",
    }

baseline_rows = []
for c in CASES:
    ev = decompose_evidence(c)
    pred = answer_from_trace(c, ev)
    rev = review_answer(pred, c.gold, ev)
    baseline_rows.append({
        "case_id": c.case_id,
        "question": c.question,
        "prediction": pred,
        "gold": c.gold,
        "timeline": " → ".join(ev["timeline"]),
        "timeline_length": len(ev["timeline"]),
        "edge_count": len(ev["all_edges"]),
        "evidence_count": len(c.evidence),
        **rev
    })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

## 4. Build stage feature vectors

In [ ]:
def decomposition_features(case: TemporalCase) -> Dict:
    qa = extract_question_anchors(case.question)
    ev = decompose_evidence(case)
    required = qa["anchors"]
    required_n = qa["required_anchor_count"]
    timeline = ev["timeline"]
    edges = ev["all_edges"]

    if required in (["timeline_start"], ["timeline_end"]):
        recovered_n = 1 if timeline else 0
    else:
        recovered_n = sum(1 for a in required if loose_contains(a, timeline))

    anchor_score = recovered_n / required_n if required_n else 1.0
    min_edges = 2 if case.answer_type == "between" else 1
    constraint_coverage = min(1.0, len(edges) / min_edges) if min_edges else 1.0
    operator_count = len(detect_operators(case.question))

    return {
        "anchor_score": anchor_score,
        "constraint_coverage": constraint_coverage,
        "recovered_edges": len(edges),
        "operator_count": operator_count,
        "timeline_length": len(timeline),
    }

def evidence_features(case: TemporalCase) -> Dict:
    ev = decompose_evidence(case)
    fact_count = len(ev["facts"])
    explicit_edge_count = len(ev["explicit_edges"])
    derived_edge_count = len(ev["derived_edges"])
    total_edges = len(ev["all_edges"])
    evidence_density = total_edges / max(1, len(case.evidence))
    return {
        "fact_count": fact_count,
        "explicit_edge_count": explicit_edge_count,
        "derived_edge_count": derived_edge_count,
        "total_edges": total_edges,
        "evidence_density": evidence_density,
    }

def review_features(case: TemporalCase, baseline_row: Dict) -> Dict:
    ambiguity_flag = 0 if baseline_row["review_status"] == "PASS" else 1
    return {
        "exact_match": baseline_row["exact_match"],
        "supported": baseline_row["supported"],
        "answer_known": baseline_row["answer_known"],
        "ambiguity_flag": ambiguity_flag,
        "evidence_density": baseline_row["edge_count"] / max(1, baseline_row["evidence_count"]),
    }

def answer_features(case: TemporalCase, baseline_row: Dict) -> Dict:
    pred = str(baseline_row["prediction"])
    gold = str(baseline_row["gold"])
    pred_len = 0 if pred == "UNKNOWN" else len(pred.split())
    gold_len = len(gold.split())
    answer_known = int(pred != "UNKNOWN")
    answer_type_id = {
        "between": 1,
        "first": 2,
        "after": 3,
        "last": 4,
        "yes_no": 5,
    }.get(case.answer_type, 0)
    return {
        "answer_known": answer_known,
        "prediction_token_length": pred_len,
        "gold_token_length": gold_len,
        "answer_type_id": answer_type_id,
        "is_yes_no": int(case.answer_type == "yes_no"),
    }

stage_rows = []
vector_store = {}

for c in CASES:
    base = baseline_df[baseline_df["case_id"] == c.case_id].iloc[0].to_dict()
    dec = decomposition_features(c)
    evf = evidence_features(c)
    rev = review_features(c, base)
    ans = answer_features(c, base)

    vector_store[c.case_id] = {
        "decomposition": np.array(list(dec.values()), dtype=float),
        "evidence": np.array(list(evf.values()), dtype=float),
        "review": np.array(list(rev.values()), dtype=float),
        "answer": np.array(list(ans.values()), dtype=float),
    }

    row = {"case_id": c.case_id}
    row.update({f"decomposition_{k}": v for k, v in dec.items()})
    row.update({f"evidence_{k}": v for k, v in evf.items()})
    row.update({f"review_{k}": v for k, v in rev.items()})
    row.update({f"answer_{k}": v for k, v in ans.items()})
    stage_rows.append(row)

features_df = pd.DataFrame(stage_rows)
features_df

## 5. Normalize vectors and compute cosine alignment

In [ ]:
def safe_norm(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    if n == 0:
        return v
    return v / n

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    aa = safe_norm(a)
    bb = safe_norm(b)
    if np.linalg.norm(aa) == 0 or np.linalg.norm(bb) == 0:
        return 0.0
    return float(np.dot(aa, bb))

ALIGNMENT_PAIRS = [
    ("decomposition", "review"),
    ("decomposition", "answer"),
    ("evidence", "answer"),
    ("review", "answer"),
]

alignment_rows = []
for case_id, stages in vector_store.items():
    for left, right in ALIGNMENT_PAIRS:
        cos_val = cosine(stages[left], stages[right])
        margin = cos_val - PHASE_LOCK_GATE
        if cos_val >= PHASE_LOCK_GATE:
            label = "phase_locked"
        elif cos_val >= PHASE_LOCK_GATE - NEAR_BOUNDARY_MARGIN:
            label = "near_boundary"
        else:
            label = "drift_candidate"
        alignment_rows.append({
            "case_id": case_id,
            "alignment_pair": f"{left}_to_{right}",
            "left_stage": left,
            "right_stage": right,
            "cosine_alignment": cos_val,
            "phase_lock_gate": PHASE_LOCK_GATE,
            "margin": margin,
            "label": label,
        })

alignment_df = pd.DataFrame(alignment_rows)
alignment_df

## 6. Case-level phase-lock classification

In [ ]:
case_class_rows = []
for case_id, group in alignment_df.groupby("case_id"):
    min_alignment = group["cosine_alignment"].min()
    mean_alignment = group["cosine_alignment"].mean()
    drift_count = int((group["label"] == "drift_candidate").sum())
    near_count = int((group["label"] == "near_boundary").sum())

    if drift_count > 0:
        case_label = "drift_candidate"
    elif near_count > 0:
        case_label = "near_boundary"
    else:
        case_label = "phase_locked"

    case_class_rows.append({
        "case_id": case_id,
        "min_alignment": min_alignment,
        "mean_alignment": mean_alignment,
        "drift_pair_count": drift_count,
        "near_boundary_pair_count": near_count,
        "case_phase_lock_label": case_label,
    })

classification_df = pd.DataFrame(case_class_rows)
classification_df = classification_df.merge(
    baseline_df[["case_id", "prediction", "gold", "review_status"]],
    on="case_id",
    how="left"
)
classification_df

## 7. Figure 1 — cosine alignment by pair

In [ ]:
plot_df = alignment_df.copy()
pair_names = list(plot_df["alignment_pair"].unique())

plt.figure(figsize=(11, 5.5))
x = np.arange(len(plot_df))
plt.bar(x, plot_df["cosine_alignment"])
plt.axhline(PHASE_LOCK_GATE, linestyle="--", linewidth=1.8, label="45° phase-lock gate")
plt.xticks(x, [f"{r.case_id}\n{r.alignment_pair.replace('_to_', '→')}" for r in plot_df.itertuples()], rotation=75, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Cosine alignment")
plt.title("Stage-to-stage cosine alignment")
plt.legend()
plt.tight_layout()

fig1_path = FIG_DIR / "03_alignment_by_pair.png"
plt.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()
fig1_path

## 8. Figure 2 — phase-lock gate matrix

In [ ]:
matrix_df = alignment_df.pivot(index="case_id", columns="alignment_pair", values="cosine_alignment")
matrix = matrix_df.to_numpy()

plt.figure(figsize=(10, 4.8))
plt.imshow(matrix, aspect="auto", vmin=0, vmax=1)
plt.colorbar(label="cosine alignment")
plt.yticks(np.arange(matrix_df.shape[0]), matrix_df.index)
plt.xticks(np.arange(matrix_df.shape[1]), [c.replace("_to_", "→") for c in matrix_df.columns], rotation=25, ha="right")
plt.title("Phase-lock alignment matrix")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        marker = "✓" if matrix[i, j] >= PHASE_LOCK_GATE else "!"
        plt.text(j, i, f"{matrix[i, j]:.2f}\n{marker}", ha="center", va="center")

plt.tight_layout()
fig2_path = FIG_DIR / "03_phase_lock_matrix.png"
plt.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()
fig2_path

## 9. Figure 3 — stable vs drift classification

In [ ]:
counts = classification_df["case_phase_lock_label"].value_counts().reindex(
    ["phase_locked", "near_boundary", "drift_candidate"],
    fill_value=0
)

plt.figure(figsize=(7.5, 4.8))
plt.bar(counts.index, counts.values)
plt.title("Case-level phase-lock classification")
plt.xlabel("Classification")
plt.ylabel("Number of cases")
plt.xticks(rotation=15)
plt.tight_layout()

fig3_path = FIG_DIR / "03_classification_counts.png"
plt.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()
fig3_path

## 10. Save artifacts

In [ ]:
features_path = ARTIFACT_DIR / "03_phase_lock_features.csv"
alignment_path = ARTIFACT_DIR / "03_phase_lock_alignment.csv"
classification_path = ARTIFACT_DIR / "03_phase_lock_classification.csv"
summary_path = DOCS_DIR / "03_phase_lock_validation_summary.md"

features_df.to_csv(features_path, index=False)
alignment_df.to_csv(alignment_path, index=False)
classification_df.to_csv(classification_path, index=False)

mean_alignment = alignment_df["cosine_alignment"].mean()
min_alignment = alignment_df["cosine_alignment"].min()
phase_locked_cases = int((classification_df["case_phase_lock_label"] == "phase_locked").sum())
drift_cases = int((classification_df["case_phase_lock_label"] == "drift_candidate").sum())

summary_md = f"""# Notebook 03 — Phase-Lock Validation

Repo: `{REPO_NAME}`  
Notebook: `notebooks/03_phase_lock_validation.ipynb`

## Purpose

This notebook measures whether adaptive temporal reasoning remains stable across decomposition, evidence, review, and answer stages.

Canonical 45° gate:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```

Numerical gate:

```text
{PHASE_LOCK_GATE:.6f}
```

## Summary metrics

| Metric | Value |
|---|---:|
| Mean cosine alignment | {mean_alignment:.3f} |
| Minimum cosine alignment | {min_alignment:.3f} |
| Phase-locked cases | {phase_locked_cases} |
| Drift-candidate cases | {drift_cases} |
| Number of cases | {len(classification_df)} |
| Number of stage-pair alignments | {len(alignment_df)} |

## Figures

- `figures/03_alignment_by_pair.png`
- `figures/03_phase_lock_matrix.png`
- `figures/03_classification_counts.png`

## Artifacts

- `artifacts/03_phase_lock_features.csv`
- `artifacts/03_phase_lock_alignment.csv`
- `artifacts/03_phase_lock_classification.csv`

## Handoff to Notebook 04

Notebook 04 should implement a residual feedback loop:

```text
detect weakest stage-pair
compute residual = gate - observed alignment
revise weakest stage
re-measure phase-lock
```
"""

summary_path.write_text(summary_md)

print(features_path)
print(alignment_path)
print(classification_path)
print(summary_path)

## 11. Handoff to Notebook 04 — residual feedback loop

In [ ]:
weakest_pairs = alignment_df.sort_values("cosine_alignment").groupby("case_id").first().reset_index()
weakest_pairs = weakest_pairs[["case_id", "alignment_pair", "cosine_alignment", "margin", "label"]]
weakest_pairs

## 12. Final notebook status

In [ ]:
status = {
    "repo": REPO_NAME,
    "notebook": NOTEBOOK_ID,
    "phase_lock_gate": PHASE_LOCK_GATE,
    "cases": len(CASES),
    "mean_alignment": round(float(mean_alignment), 3),
    "min_alignment": round(float(min_alignment), 3),
    "phase_locked_cases": phase_locked_cases,
    "drift_candidate_cases": drift_cases,
    "artifacts": [
        str(features_path),
        str(alignment_path),
        str(classification_path),
        str(summary_path),
        str(fig1_path),
        str(fig2_path),
        str(fig3_path),
    ],
}
print(json.dumps(status, indent=2))